# Spherical geometry — Phase 1

CPU-only trajectory, metric-baseline selection, ERP sampling, and rotation-compensated feature diagnostics. No model inference or depth accuracy claims.

Install in a dedicated Python 3.11 environment:
`python -m pip install -r experiments/spherical_geometry/requirements.txt`

Run all cells in order. Outputs go to a new directory under `outputs/`; input images and poses remain unchanged.

In [ ]:
from pathlib import Path
import sys
from datetime import datetime, timezone
from IPython.display import display
ROOT = Path.cwd().resolve()
if ROOT.name == "notebooks":
    ROOT = ROOT.parent
assert (ROOT / "experiments/spherical_geometry/run.py").is_file()
sys.path.insert(0, str(ROOT))
from experiments.spherical_geometry.run import Config, run


## Experiment configuration

Baseline bins are half-open intervals in metres, evaluated over all distinct unordered frame pairs. Empty bins remain empty. Sampling is deterministic and uniform within each bin. Pair counts are not independent sample counts because frames can be reused.

The image diagnostic uses a small, reproducible subset per bin. Increase it after inspecting runtime and coverage. SIFT excludes poles and the seam; a missing measurement is not zero parallax.

In [ ]:
# Experiment settings
SEQUENCE = ROOT / "Standard_GT/city01_route01"
if not SEQUENCE.exists() and (ROOT / "standard_GT/city01_route01").exists():
    SEQUENCE = ROOT / "standard_GT/city01_route01"
stamp = datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%S_%fZ")
CONFIG = Config(
    sequence=str(SEQUENCE),
    output=str(ROOT / "outputs/spherical_geometry" / stamp),
    baseline_edges_m=(0., .01, .1, .5, 1., 2., 5., 10.),
    pairs_per_bin=10,
    max_rotation_deg=180.,
    seed=17,
    image_diagnostics=True,
    diagnostic_pairs_per_bin=2,
    latitude_bin_width_deg=15.,
    static_formats=("png", "pdf"),
)
display(CONFIG)


## Run and save raw measurements

This reads the full pose sequence without decoding all images. Pair selection has quadratic time but linear working memory. Only selected diagnostic images are decoded, with a bounded cache.

The repository's `source_matlab_v1` convention is preserved: camera-to-world transforms, metres, right/down/forward camera axes, endpoint-inclusive ERP. Capture alignment remains unverified; inspect notebook 05 before interpreting residuals as physical parallax.

In [ ]:
result = run(CONFIG)
print("Saved:", result["output"])
display(result["statistics"]["summary"])
display(result["pairs"])
import pandas as pd
display(pd.DataFrame(result["coverage"]))


## Camera trajectory, translation, rotation and baseline distributions

Adjacent-step statistics use retained-frame order. Selected-pair distributions use baseline-stratified sampling and are not estimates of the all-pair distribution. Bin coverage reports exact eligible pair counts within the configured baseline and rotation ranges.

In [ ]:
for name in ("trajectory", "step_distance_m", "rotation_step_deg", "baseline_coverage", "baseline_cdf"):
    result["figures"][name].show()


## ERP latitude and solid-angle sampling

The map shows exact integration-cell areas around the endpoint grid. The duplicate seam samples each receive half-width cells. Polar samples receive spherical-cap cells. Total solid angle is 4π steradians. This is sampling geometry, not model error.

In [ ]:
result["figures"]["erp_solid_angle"].show()
display(pd.read_csv(result["output"] / "spherical_rows.csv"))


## Rotation-compensated motion diagnostics

For matched unit bearings `a` and `b`, the residual angle is between `a` and `R_wc_i.T @ R_wc_j @ b`. Rotations are read from the recorded poses and are not fitted to minimize residuals. Mutual SIFT ratio-test matches are tentative; there is no residual-based rejection that could suppress true translation parallax.

Residuals can include translation, incorrect matches, moving objects, noncentral stitching, and pose errors. Latitude profiles are feature-sampled diagnostic profiles, not area-weighted reconstruction errors. Sparse maps show observed matches only; no dense values are invented.

In [ ]:
display(result["diagnostics"])
for name in ("sparse_parallax_map", "latitude_residual_profile", "baseline_parallax"):
    if name in result["figures"]:
        result["figures"][name].show()
    else:
        print(name + ": no measurements available.")


## Export status and review gate

CSV tables, exact pair/frame manifests, configuration, provenance and HTML figures have been saved. PNG/PDF exports require Kaleido; failures are explicitly listed below. No depth errors or model rankings are produced.

Before Phase 2: review the pose/image alignment diagnostic and coverage, supply verified scene-distance/depth data if available, and define each model's camera and depth conventions. Reuse these frame manifests with existing inference orchestration, record all pixel transformations, and report metric-scale evaluation separately from scale-aligned evaluation.

In [ ]:
display(pd.DataFrame(result["figure_exports"]))
failed = [row for row in result["figure_exports"] if row["status"] == "failed"]
if failed:
    print("Static exports failed. Inspect figure_exports.json; CSV and HTML outputs remain available.")
print("Phase 2 is deferred pending validation and review.")
